# MLP baseline on the shared Phase 1 protocol
This rerun keeps the original teammate notebook intact. It uses the shared eight-candidate, ten-feature arrays and scorer, and executes all five fixed trip folds. The 64/32 architecture and 25-epoch budget follow the original experiment. Shared fold-fitted scaling is used, with early stopping disabled to avoid an internal random candidate split.


In [1]:
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
assert (ROOT / "data").is_dir(), "Open this notebook from the repository root or notebooks directory"
os.chdir(ROOT)


In [2]:
get_ipython().run_line_magic("run", "notebooks/00_shared_training.ipynb")

In [3]:
MODEL = "mlp"
RUN_ID = "shared_mlp_baseline_01"

# Match the teammate’s architecture and 25-epoch budget. Use the outer trip
# folds for evaluation and disable internal random-row early stopping.
_shared_factory = make_model
def make_model(name):
    model = _shared_factory(name)
    if name == "mlp":
        model.set_params(mlpclassifier__batch_size=256, mlpclassifier__max_iter=25)
    return model


## Optional smoke check
Set RUN_SMOKE=True to check the interface on 200 observations. These are not full experiment results.

In [4]:
RUN_SMOKE = False
if RUN_SMOKE:
    run(MODEL, "processed", f"scratch/{RUN_ID}", folds=[0], final=True, smoke=True)

## Full model comparison
This trains five trip-based CV fits and a final all-training fit, then evaluates student-test data. Preserve warnings and configuration. Use a fresh RUN_ID for another experiment.

In [5]:
run(MODEL, "processed", f"results/{RUN_ID}", folds=[0,1,2,3,4], final=True)

fold_0 {"split": "fold_0", "fit_seconds": 76.89530572299554, "inference_seconds": 0.769773970998358, "inference_ms_per_observation": 0.02682139271771282, "metrics": {"all_points": {"observations": 28700, "top1_accuracy": 0.7670034843205575, "macro_precision": 0.6296589033976571, "macro_recall": 0.5674588491734761, "macro_f1": 0.5861745694458836, "candidate_miss_rate": 0.00975609756097561, "categorical_log_loss_with_miss_state": 0.9882306076306351}, "reliable_points": {"observations": 20087, "top1_accuracy": 0.8635435854034949, "macro_precision": 0.7762706605337828, "macro_recall": 0.733715726572523, "macro_f1": 0.7492318013427043, "candidate_miss_rate": 0.0010952357245979987, "categorical_log_loss_with_miss_state": 0.37756524986561985}, "reliable_coverage": 0.6998954703832753, "candidate_binary_log_loss": 0.16963016718154372, "route_continuity": null, "route_continuity_reason": "Needs raw road network and an agreed path/connectivity definition; do not invent a value."}, "fit_subset_met

## Inspect measured results

In [6]:
import json
output = Path("results") / RUN_ID / MODEL
print((output / "summary.json").read_text())
records = json.loads((output / "metrics.json").read_text())
for record in records:
    print(record["split"], record["metrics"]["all_points"], record["warnings"])

{
  "model": "mlp",
  "smoke": false,
  "cv_macro_f1_mean": 0.6432068154883326,
  "cv_macro_f1_std": 0.048618735911071256,
  "completed_folds": 5,
  "student_test_completed": true
}
fold_0 {'observations': 28700, 'top1_accuracy': 0.7670034843205575, 'macro_precision': 0.6296589033976571, 'macro_recall': 0.5674588491734761, 'macro_f1': 0.5861745694458836, 'candidate_miss_rate': 0.00975609756097561, 'categorical_log_loss_with_miss_state': 0.9882306076306351} ["Stochastic Optimizer: Maximum iterations (25) reached and the optimization hasn't converged yet."]
fold_1 {'observations': 40694, 'top1_accuracy': 0.752592519781786, 'macro_precision': 0.723411795605744, 'macro_recall': 0.6760621779732813, 'macro_f1': 0.6907680102383281, 'candidate_miss_rate': 0.10217722514375584, 'categorical_log_loss_with_miss_state': 3.122959978880735} ["Stochastic Optimizer: Maximum iterations (25) reached and the optimization hasn't converged yet."]
fold_2 {'observations': 49146, 'top1_accuracy': 0.46791193586

## Your interpretation
Write your own explanation of the algorithm/objective, hyperparameters, actual results, timing scope, fit and generalisation, convergence where applicable, and proxy-label limitations. Route continuity remains N/A until common graph scoring is implemented. Do not invent loss curves or results.